In [1]:
import json
import os

import numpy as np
import polars as pl
from google.colab import drive
from sentence_transformers import SentenceTransformer

# Upgraded to MPNet 768-dim (was granite-30m 384-dim). Trade-offs:
# - 2x output dimension means 2x storage for book_embeddings, 2x memory in MPS during training,
#   and downstream feature dims double (item: 395 -> 779, user: 779 -> 1547).
# - ~3x slower per encode on T4 vs granite-30m, but T4 fits fp16 MPNet fine.
# - Expected +20-30% on HR@10 according to v3-overhaul-deferred-items.md.
# Alternatives if MPNet is too slow or you want even more: BAAI/bge-base-en-v1.5 (768-dim, similar speed),
# BAAI/bge-large-en-v1.5 (1024-dim, ~3x slower).
EMBEDDING_MODEL = "sentence-transformers/all-mpnet-base-v2"

# Output naming — _v2 suffix preserves the original 384-dim files. Downstream code that expects
# book_embeddings.npy will keep working until you point it at book_embeddings_v2.npy explicitly.
EMBEDDING_VERSION = "v2"  # bump if you change the model again

In [2]:
drive.mount("/content/drive")  # Mount Google Drive to access data files

Mounted at /content/drive


In [3]:
# Load filtered book parquet from Google Drive
books_df = pl.read_parquet("/content/drive/MyDrive/books_with_genres.parquet")

In [4]:
print(len(books_df))

1782579


In [ ]:
# Load embedding model
model = SentenceTransformer(EMBEDDING_MODEL, device="cuda")

# fp16 inference — works on T4 for both granite and MPNet, halves memory + speeds up encode
# without measurable quality loss for sentence-similarity tasks.
model.half()

# Versioned output paths: separate shard dir + embedding file per model version so re-running
# with a new model doesn't overwrite the previous results.
shard_dir = f"/content/drive/MyDrive/embeddings_{EMBEDDING_VERSION}"
output_path = f"/content/drive/MyDrive/book_embeddings_{EMBEDDING_VERSION}.npy"
os.makedirs(shard_dir, exist_ok=True)

# MPNet is ~3x slower per token than granite-30m. Lower batch size if memory is tight,
# but T4 fits 256 fp16 MPNet samples comfortably.
BATCH_SIZE = 256
SHARD_SIZE = 50000

# Create rows of books with concatenated title and description - handle nulls
books_df = books_df.with_columns(
    pl.when(pl.col("description").is_null() | (pl.col("description") == ""))
    .then(pl.col("title"))
    .otherwise(pl.col("title") + " - " + pl.col("description"))
    .alias("text")
)

num_books = len(books_df)
for shard_idx in range(0, num_books, SHARD_SIZE):
    shard_path = f"{shard_dir}/shard_{shard_idx:08d}.npy"
    if os.path.exists(shard_path):
        print(f"Shard {shard_idx} already exists, skipping...")
        continue
    shard_books = books_df[shard_idx : shard_idx + SHARD_SIZE]
    shard_texts = shard_books["text"].to_list()
    shard_embeddings = model.encode(
        shard_texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    np.save(shard_path, shard_embeddings)
    print(f"Saved shard {shard_idx} ({shard_embeddings.shape}) to {shard_path}")

# Concatenate all shards into a single matrix and save back to Drive
all_shard = [np.load(f"{shard_dir}/{f}") for f in sorted(os.listdir(shard_dir)) if f.startswith("shard_")]
embeddings = np.concatenate(all_shard, axis=0)
print(f"Final embeddings shape: {embeddings.shape}, dtype: {embeddings.dtype}")
np.save(output_path, embeddings)
print(f"Saved final embeddings to {output_path}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/196 [00:00<?, ?it/s]

Saved shard 1300000 ((50000, 768)) to /content/drive/MyDrive/embeddings_v2/shard_01300000.npy


Batches:   0%|          | 0/196 [00:00<?, ?it/s]

Saved shard 1350000 ((50000, 768)) to /content/drive/MyDrive/embeddings_v2/shard_01350000.npy


Batches:   0%|          | 0/196 [00:00<?, ?it/s]

In [ ]:
# Save book_id ↔ row-index mapping. This is INDEPENDENT of the embedding model — same row order
# applies to both the granite-384 and MPNet-768 embeddings. Only re-save if you change books_df.
book_id_to_index = {row["book_id"]: idx for idx, row in enumerate(books_df.select("book_id").to_dicts())}
with open("/content/drive/MyDrive/book_id_to_index.json", "w") as f:
    json.dump(book_id_to_index, f, indent=2)

# Reverse mapping for index -> book_id (used during inference/eval to look up metadata)
index_to_book_id = {idx: row["book_id"] for idx, row in enumerate(books_df.select("book_id").to_dicts())}
with open("/content/drive/MyDrive/index_to_book_id.json", "w") as f:
    json.dump(index_to_book_id, f, indent=2)

: 

In [ ]:
# Optional: round-trip verification — load the embeddings + index from Drive, save to local
# data/transformed/ so you can spot-check shapes and re-use them locally without re-downloading.
# Skip if you're running purely on Colab.

with open("/content/drive/MyDrive/book_id_to_index.json") as f:
    book_id_to_index = json.load(f)
os.makedirs("data/transformed", exist_ok=True)
with open("data/transformed/book_id_to_index.json", "w") as f:
    json.dump(book_id_to_index, f, indent=2)

# Reload + concat shards from Drive, save locally with the versioned name.
all_shard = [np.load(f"{shard_dir}/{f}") for f in sorted(os.listdir(shard_dir)) if f.startswith("shard_")]
embeddings = np.concatenate(all_shard, axis=0)
local_path = f"data/transformed/book_embeddings_{EMBEDDING_VERSION}.npy"
np.save(local_path, embeddings)
print(f"Saved {embeddings.shape} {embeddings.dtype} to {local_path}")